# zReg — Basics Tutorial

This notebook introduces the core zReg workflow: loading a time-series of point clouds
from a `.tracklets` file, inspecting the per-frame data structure, and visualising two
frames side-by-side with matplotlib. It is designed as a first contact with the
library and requires no cluster access — just set `DATA_PATH` to your local file.

In [ ]:
DATA_PATH = "# Replace with the path to your .tracklets file"

import zreg
import copy
import numpy as np

## Load trajectory data

`load_data_from_tracklets` reads a `.tracklets` binary file and returns a list of
per-frame point cloud dictionaries. Each dictionary contains `"pos"` (3-D coordinates),
`"label"` (per-point label), and `"id"` (cell-track identifiers). The `device` argument
controls whether tensors are allocated on CPU or GPU.

In [ ]:
pcs, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH,
    device="cpu",
)
print(pcs[0]["pos"].shape)
print(pcs[0]["label"].shape)
print(pcs[0]["id"].shape)

## Visualisation helper

`draw_registration_result` accepts two `zRegPointCloud` dictionaries and renders them
as a 3-D matplotlib scatter plot, colouring the source gold and the target blue.
This makes it straightforward to compare a source and target frame before and after
registration, with no external rendering dependencies beyond matplotlib.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt


def _to_numpy(pc):
    """Extract an Nx3 position array from a zRegPointCloud or bare tensor."""
    pos = pc["pos"] if isinstance(pc, dict) else pc
    return pos.detach().cpu().numpy() if hasattr(pos, "detach") else pos


def draw_registration_result(source, target=None):
    """Render source (gold) and optionally target (blue) clouds in a 3-D scatter."""
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    src = _to_numpy(source)
    ax.scatter(src[:, 0], src[:, 1], src[:, 2],
               c="#FFB300", s=1.5, alpha=0.8, linewidths=0, label="Source")
    if target is not None:
        tgt = _to_numpy(target)
        ax.scatter(tgt[:, 0], tgt[:, 1], tgt[:, 2],
                   c="#1976D2", s=1.5, alpha=0.8, linewidths=0, label="Target")
        ax.legend(markerscale=8)
    ax.set_xlabel("x", fontsize=8)
    ax.set_ylabel("y", fontsize=8)
    ax.set_zlabel("z", fontsize=8)
    plt.tight_layout()
    plt.show()


## Visualise two frames

Here we display `pcs[0]` (the first frame) and `pcs[100]` (the 101st frame) of the
loaded trajectory. Rendering these two frames together gives a quick visual impression
of how much cell positions change over 100 time steps — a useful sanity check before
running any registration algorithm.

In [ ]:
draw_registration_result(pcs[0], pcs[100])